db.py


In [5]:
#pip install "fastapi"

 Nadpisanie metody _conn, żeby użyć poprawnego połączenie z bazą w pamięci

In [ ]:
import sqlite3
from db import EPCRepository

def fixed_conn(self):
    if not hasattr(self, '_shared_conn'):
        self._shared_conn = sqlite3.connect(self._path, check_same_thread=False)
        self._shared_conn.row_factory = sqlite3.Row
    return self._shared_conn

# Podpinamy poprawkę do klasy
EPCRepository._conn = fixed_conn

## Unitests

In [7]:
import unittest
import sqlite3
import os
from db import EPCRepository
from models import UEState

class TestEPCRepository(unittest.TestCase):
    
    def setUp(self):
        """Uruchamia się przed każdym testem. Tworzy nową bazę w RAM."""
        # Używamy :memory: dla maksymalnej izolacji i szybkości
        self.repo = EPCRepository(db_path=":memory:")

    def test_attach_ue_and_exists(self):
        """Test czy UE poprawnie się dodaje."""
        ue_id = 10
        self.repo.attach_ue(ue_id)
        self.assertTrue(self.repo.ue_exists(ue_id))
        
        # Test duplikatu (czy rzuca błąd)
        with self.assertRaises(ValueError):
            self.repo.attach_ue(ue_id)

    def test_default_bearer_9_creation(self):
        """Test czy automatycznie powstaje bearer 9 ."""
        ue_id = 5
        self.repo.attach_ue(ue_id)
        
        state = self.repo.get_ue(ue_id)
        self.assertIn(9, state.bearers, "Bearer 9 powinien być dodany automatycznie!")
    
    def test_duplicate_ue_raises_error(self):
        """Test czy próba duplikatu UE rzuca błąd."""
        self.repo.attach_ue(10)
        with self.assertRaises(ValueError):
            self.repo.attach_ue(10)

    def test_duplicate_bearer_raises_error(self):
        """Test czy próba dodania istniejącego bearera rzuca błąd."""
        ue_id = 10
        self.repo.attach_ue(ue_id)
        # Bearer 9 jest dodawany automatycznie, więc dodanie go ponownie powinno wywalić błąd
        with self.assertRaises(ValueError):
            self.repo.add_bearer(ue_id, 9)

    def test_detach_ue_removes_it(self):
        """Test czy detach faktycznie usuwa UE z bazy."""
        self.repo.attach_ue(10)
        self.repo.detach_ue(10)
        self.assertFalse(self.repo.ue_exists(10))

    def test_delete_missing_bearer_raises_error(self):
        """Test czy usuwanie nieistniejącego bearera rzuca błąd."""
        self.repo.attach_ue(10)
        with self.assertRaises(ValueError):
            self.repo.delete_bearer(10, 5) 

    def test_delete_default_bearer_raises_error(self):
        """Test czy usuwanie domyślnego bearera 9 jest zablokowane."""
        self.repo.attach_ue(10)
        with self.assertRaises(ValueError):
            self.repo.delete_bearer(10, 9)

    def test_operation_on_nonexistent_ue_raises_error(self):
        """Test czy operacje na nieistniejącym UE rzucają błąd."""
        with self.assertRaises(ValueError):
            self.repo.get_ue(999)
        with self.assertRaises(ValueError):
            self.repo.detach_ue(999)

    def test_list_ues_shows_all(self):
        """Test czy list_ues pokazuje poprawne ID."""
        self.repo.attach_ue(1)
        self.repo.attach_ue(2)
        self.repo.attach_ue(3)
        
        ues = list(self.repo.list_ues())
        self.assertEqual(len(ues), 3)
        self.assertIn(1, ues)
        self.assertIn(2, ues)
        self.assertIn(3, ues)

    def test_reset_functionality(self):
        """Test czy reset usuwa wszystko."""
        self.repo.attach_ue(1)
        self.repo.attach_ue(2)
        
        self.repo.reset_all()
        
        self.assertFalse(self.repo.ue_exists(1))
        self.assertFalse(self.repo.ue_exists(2))

# Uruchomienie testów
if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..........
----------------------------------------------------------------------
Ran 10 tests in 0.026s

OK
